# 00. Notation and reading guide

**Read this notebook first, and keep it open while you read the others.** It defines every abbreviation the series uses, maps the three notations in play (thesis `residual.tex`, the simulator code, the Guha-Annaswamy paper) and gives one complete scalar MRAC derivation that all later derivations imitate.

## 1. Purpose of the series

The notebooks have two jobs.

1. **Class tutorials.** NB01-NB04 teach Model Reference Adaptive Control (MRAC) and the Lyapunov argument behind it, from the MIT rule to the MRAC-RL framework of Guha and Annaswamy.
2. **Research log (bitácora).** NB05-NB13 record, step by step, an offline study of a **single planar quadrotor carrying a cable-suspended payload**. They show what was derived, what was run, and exactly what each result does and does not support.

Status of the second part: **EXPLORATORY, offline, PROXY.** The results come from a reduced Newton-Euler model on the development PC. They are not Pymunk-simulator results, not a stability proof for the sampled nonlinear loop, not hardware results and not a pre-registered study. Each notebook repeats its own claim limits.

**The goal of every derivation:** a student who has read the earlier notebooks can follow each step without opening the thesis repository. The thesis derivation note (called *MQP*, see the glossary) stays the reference source, but each notebook carries the key steps itself.

## 2. Series map and reading order

| Notebook | Question it answers | Scope |
|---|---|---|
| **NB00** (this one) | What do the abbreviations and symbols mean, and how do the three notations map onto each other? | reference |
| **NB01** Introduction to MRAC | If a plant has unknown parameters, how can the controller adapt its gains online so that the plant follows a reference model? (gradient "MIT rule") | generic theory |
| **NB02** MRAC meets Lyapunov | How does a Lyapunov function replace the heuristic MIT rule, and what changes when a matched disturbance with a chosen basis $\Phi$ is added? | generic theory, scalar |
| **NB03** Lyapunov, general system | The same argument for vectors and matrices: unknown control effectiveness $\Lambda$ and trace identities | generic theory, MIMO |
| **NB04** RL and MRAC | How does the MRAC-RL framework of Guha and Annaswamy use an RL policy's closed loop as the reference model, and why does the error feedback $\xi$ make $A_H$ the error matrix? | generic theory + paper |
| **NB05** Slung-load model validation | What is the taut point-load model, and does it pass its physical checks? | plant, $d = 0.3795$ m, code angle |
| **NB06** Baselines | How do PID+FF (arm 1) and the linear swing damper (arm 2) behave, and where does the torque-to-swing zero pair come from? | plant, $d = 0.3795$ m, code angle |
| **NB08** Vehicle mismatch grid | How much do vehicle parameter errors (mass, inertia, arm, motor asymmetry) hurt arms 1 and 2? | plant, $d = 0.3795$ m, code angle |
| **NB07** Arm 3 | What is the A0 / B3 route-(b) adaptive law (arm 3), and what did its first run show? (the preliminary run is superseded by its section 10) | plant, $d = 0.3795$ m, code angle |
| **NB09** Nominal-load reference | What changes when the A0 reference carries the nominal payload, and why does the law struggle at long horizons? | plant, $d = 0.3795$ m, code angle |
| **NB10** Vanilla MRAC (Guha) | Can the Guha-Annaswamy pattern be applied to the hover-linearised outer model, with an exact Lyapunov statement on that model? | derivations at $d = 0$, residual.tex angle; hover check and runs on the plant with $d = 0.3795$ m |
| **NB11** Extension (a) | What changes when the outer model is the exact nonlinear taut model in cable-frame channels? | $d = 0$, residual.tex angle |
| **NB12** Extension (a)+(b) | What does adding disturbance-estimator weights add to the Lyapunov storage? | $d = 0$, residual.tex angle |
| **NB13** Extension (a)+(b)+(c) | What does the $\sigma$ leak add, and what does it cost? | $d = 0$, residual.tex angle |

**Reading order and prerequisites.**

```text
NB00 --> NB01 --> NB02 --> NB03 --> NB04 -----------+
                                                    v
NB05 --> NB06 --> NB08 --> NB07 --> NB09          NB10 --> NB11 --> NB12 --> NB13
```

- NB01-NB04 are generic theory: read them in order. NB04 builds on NB02 and NB03.
- NB05-NB09 form the **logbook branch** (the A0 law at $d = 0.3795$ m): NB05 gives the plant, NB06 the baselines, NB08 the mismatch grid, NB07 and NB09 the A0 law.
- Read the logbook branch in the order NB05, NB06, NB08, NB07, NB09: NB05, NB06, NB08 and NB07 are logbook steps 1 to 4, and NB09 follows NB07. NB08 comes before NB07 because it was added after the 2026-11-24 meeting as step 3 of 4. In the diagram the branch arrows show this reading order; for prerequisites, NB08 and NB07 both build on NB06, and NB09 builds on NB07.
- NB10-NB13 form the **vanilla branch** (the Guha pattern, derived at $d = 0$; NB10's hover check and runs use the plant with $d = 0.3795$ m, NB11-NB13 run at $d = 0$, except their robust readout at $d = 0.3795$ m): NB10 needs NB04 (theory) and the hover linearisation of the plant (NB05, or MQP §5.7). NB11-NB13 each need the previous one.

**Which model, which angle.** Two plant scopes are used. Do not mix their formulas.

| Notebooks | Cable attachment offset $d$ | Rope angle in the formulas |
|---|---|---|
| NB05-NB09 | $d = 0.3795$ m (canonical offset plant) | **code angle** $\phi_{\rm code}$; the simple sign flip to residual.tex does **not** apply |
| NB10-NB13 derivations; NB11-NB13 runs (except the robust readout, $d = 0.3795$ m) | $d = 0$ (cable at the vehicle centre of mass) | residual.tex angle $\varphi = -\phi_{\rm code}$ |
| NB10 hover check and runs (Sections 5, 7, 8) | $d = 0.3795$ m (canonical offset plant) | the offset coupling is a residual outside the $d = 0$ derivation |

## 3. Glossary

### 3.1 Project terms

| Term | Meaning | Source |
|---|---|---|
| **MQP** | Short name of the thesis derivation note *MRAC for a planar quadrotor with a cable-suspended payload* (**M**RAC, **Q**uadrotor, **P**ayload), file `.ai/research/derivations/mrac-quadrotor-payload.md` in the thesis repository. Status: **PROPOSED**, not accepted. "MQP §6.14" means section 6.14 of that file. You do not need the file: the notebooks carry the key steps. | MQP §0.1 |
| **EQ-MR-xx** | Row ID in the thesis equation registry (`.ai/research/equation-registry.md`), section "MR: MRAC offline reduced model (PROPOSED, research-only)". For example EQ-MR-22 is the vanilla MRAC law (MQP §6.14) and EQ-MR-23 the nonlinear-model law (MQP §6.15). | MQP §0, §0.1 |
| **residual.tex** | The author's thesis chapter `residual.tex` (a local, untracked draft; lines 95-292). Its notation is **primary**; the notebooks write its symbols and convert from the code with the notation map of section 4. | notation note §0 |
| **code coordinates / code angle** | The symbols of the offline simulator code and of MQP: $z$ up, the rope angle $\phi_{\rm code}$ measured from the downward vertical, rope direction $q = [\sin\phi_{\rm code}, -\cos\phi_{\rm code}]^\top$. At $d = 0$: $\varphi_{\rm tex} = -\phi_{\rm code}$. | notation note §3 |
| **$d$** | Distance from the body centre to the cable attachment point. NB05-NB09 use $d = 0.3795$ m. NB10-NB13 derive at $d = 0$; NB10's hover check and runs use $d = 0.3795$ m, and the NB11-NB13 runs use $d = 0$, except their robust readout ($d = 0.3795$ m). (An earlier text said that NB10-NB13 use $d = 0$.) | MQP §0.1, §2.2 |
| **A0** | Architecture A0 of the MRAC plan: a nominal **reference** closed loop and the uncertain **plant** are simulated in parallel, and an adaptive law makes the plant follow the reference. | MQP §1.2 |
| **B3** | Blocker 3, the "corrected law", of the MRAC plan. Its revision defines the adaptive law of MQP §4 and §6; "B3 section n" cites section n of that revision. | MQP §0.1 |
| **route (b)** | The derivation route the MRAC plan selected for the taut reduced model: an inertia-weighted Lyapunov derivation on the vehicle coordinates. | MQP §0.1, §6.2 |
| **W1** | No-load planar quadrotor `Quad` and its adaptive law. | MQP §3-4 |
| **W2a** | Taut point-load plant `QuadLoadPend` (the model of NB05). | MQP §5 |
| **W2b** | Payload-aware adaptive law (the arm-3 law of NB07). | MQP §6.1-6.12 |
| **W2c** | Offline baselines: PID+FF and the linear damper (code, NB06). | NB06 |
| **W2d** | A0 reference that carries the nominal payload (NB09). | MQP §6.13 |
| **W2e** | Vanilla Guha-Annaswamy MRAC (NB10). | MQP §6.14 |
| **extension a / b / c** | (a) nonlinear-model MRAC at $d = 0$ (NB11); (b) plus disturbance weights (NB12); (c) plus $\sigma$-modification (NB13). | MQP §6.15, NB11-NB13 |
| **arm 1, arm 2, arm 3** | The three experimental arms of the notebooks. Arm 1: PID+FF with a mass-blind feedforward (it assumes the 4 kg nominal payload). Arm 2: PID+FF plus a fixed linear swing law (the softer law or the damper, see the next row). Arm 3: arm 2 plus the MRAC law. | NB06, NB07 |
| **PID+FF** | The baseline controller: PID feedback plus feedforward, re-implemented offline in `experiments/mrac/baselines.py`. "Mass-blind" means the feedforward does not know the true payload mass. | NB06 |
| **linear damper / softer law** | A fixed linear swing law added on the torque channel, from the package's `LinearSwingResidual`. $G(a, b)$ are its gains on $(\phi_{\rm code}, \dot\phi_{\rm code})$: the *softer law* is $G(-1,-0.5)$ and the *damper* is $G(-1,-2)$. | NB06, notation note §7 |
| **protocol 08, AM1** | Protocol 08 is the written offline experiment protocol (a local, untracked file; approved 2026-11-25) behind the cached grids of NB11-NB13. AM1 is its amendment 1, dated 2026-11-25 and written before any protocol-08 data, which fixed the gains of the nonlinear-model law. | NB11 header; protocol 08 (local, untracked) |
| **Codex NN** | A tag such as "Codex 11 F7" in the thesis notes cites finding F7 of review report number 11 of one work wave, written by the Codex AI reviewer. Report numbers restart in each wave. The reports are local, untracked files. | MQP §6.14.1; the notation note |
| **PROXY** | A plant or learner result produced on the development PC. It is a proxy for, not a replacement of, the canonical training-PC result. | AGENTS.md; workflow §4.1 |
| **EXPLORATORY** | A logbook-level result: not pre-registered, not a study, not a claim. "These notebooks are an exploratory logbook." | MRAC_RL_NOTEBOOKS.md |
| **Evidence labels** | **VERIFIED**: observed directly in a cited source, config, executed check or artifact. **INFERRED**: reasoned from verified evidence. **PROPOSED**: a future decision, not approved. **NOT VERIFIED**: identity or theorem scope not checked in the primary source. **UNKNOWN**: the evidence is missing. | AGENTS.md |

### 3.2 Control-theory terms

| Term | Meaning | Where used |
|---|---|---|
| **Reference model** | A known system $\dot x_m = A_m x_m + B_m r$ (or $\dot x_r = A_r x_r + B_r u_r$ in Guha's notation) whose response is the behaviour we want from the plant. | NB01 |
| **Hurwitz** | A square matrix is Hurwitz when every eigenvalue has a strictly negative real part. Then $\dot e = A_m e$ has $e(t) \to 0$. In the scalar case $A_m < 0$. | NB01, NB04 |
| **Lyapunov function** | A positive-definite "storage" $V$ ($V(0) = 0$, $V > 0$ elsewhere) whose derivative along trajectories is $\le 0$. It is a mathematical device, not necessarily physical energy. | NB02 |
| **Lyapunov equation** | $P A_H + A_H^\top P = -Q$: for Hurwitz $A_H$ and $Q = Q^\top > 0$ it has a unique solution $P = P^\top > 0$. $P$ weights the tracking error in $V = e^\top P e + \cdots$. | NB03, NB04 |
| **Barbalat's lemma** | If a function $f(t)$ has a finite limit as $t \to \infty$ and $\dot f$ is uniformly continuous, then $\dot f(t) \to 0$ (Khalil, *Nonlinear Systems*, 3rd ed., Lemma 8.2, printed p. 323, applied to $\dot f$). Used with $f = V$: $V$ is non-increasing and $\ge 0$, so it converges, and $\dot V = -e^\top Q e$ then tends to 0, hence $e \to 0$. It needs $\dot V$ uniformly continuous, which follows when $\ddot V$ is bounded. | NB02-NB04 |
| **PE (persistent excitation)** | A regressor $\zeta(t)$ is persistently exciting if there are $T, \alpha > 0$ with $\int_t^{t+T}\zeta\zeta^\top\,d\tau \succeq \alpha I$ for all $t$. It is a standard sufficient route to parameter convergence under an applicable identification theorem (a scalar example: Ioannou and Sun, PDF p. 164). Barbalat gives $e \to 0$ **without** parameter convergence. | NB02, NB10 |
| **UUB** | *Uniformly ultimately bounded*: after some finite time the state stays inside a ball of fixed radius, uniformly in the initial time (Khalil, Def. 4.6, printed p. 169). It permits a nonzero residual error. | NB10, NB13 |
| **$\sigma$-modification** | A leak term added to an adaptive update. The notebooks use $-\sigma(\hat K - K_0)$, a **shifted** $\sigma$-modification (leakage toward a nominal value $K_0$). Ioannou and Sun's fixed $\sigma$-modification (eq. 8.4.11, PDF pp. 571-572) leaks toward $0$; leakage toward a prior estimate is their "shifted leakage" (Remark 8.4.1(ii), PDF p. 578), whose analysis they leave to the reader. For the continuous-time, exactly matched **linear** model the notebooks state an ultimate bound $\limsup V\le\sigma\,\lVert K^*-K_0\rVert_W^2/c$ instead of $e\to0$ (NB10); that inequality is this repository's own derivation (MQP §6.14.6). It does **not** cover the sampled nonlinear extensions or the combined abc law (NB13); their behaviour is an empirical question. | NB10, NB13; MQP §6.14.6 |
| **Projection** | Keeping an estimate inside a closed convex set that contains the ideal value. In continuous time, gradient projection keeps the properties of the unprojected law and keeps the estimate in the set, if the set is convex and both the initial value and the ideal value lie in it (Ioannou and Sun, Thm. 4.4.1, PDF pp. 218-219; convexity required, Remark 4.4.1). That is an on-line parameter-estimation result, not an MRAC closed-loop theorem; in the derivations of MQP §4.6 and §6.14.6 the Lyapunov derivative of the continuous projected law then satisfies $\dot V\le 0$ (an inequality instead of the exact cancellation). The sampled code applies a componentwise clip: with positive diagonal gains $\gamma_i$ and a box containing the ideal $x^*$, the clip never increases $\sum_i (x_i-x^*_i)^2/\gamma_i$. That says nothing about the rest of the sampled step: no discrete Lyapunov decrease follows. | MQP §4.6, §6.14.6 |
| **Matched uncertainty** | Uncertainty that enters through the same channel as the control input, so the controller can cancel it directly: $\dot x = Ax + B[u + \Delta(x)]$. | NB02, NB03; MQP §1.3 |
| **Unmatched uncertainty** | Uncertainty that cannot be reached instantaneously through $B$ (for example the cable force perpendicular to the thrust, the swing dynamics, or a rope-length mismatch). | MQP §1.3 |
| **Adaptation gain $\Gamma$, $\gamma$** | Positive (matrix) gain that sets how fast an estimate moves; it sits in the denominator of the matching term of $V$. | NB02-NB04 |

## 4. Notation map: residual.tex, code, Guha

The thesis notation file `notation-residual-tex.md` is the source of this table. Scope: **cable at the vehicle centre of mass, $d = 0$.**

| residual.tex | Meaning | MQP / code | Guha |
|---|---|---|---|
| $\{W\}$, $e_1, e_2$ | inertial frame, $z$ up, gravity $-g\,e_2$ | $e_y, e_z$ | |
| $m_q$, $J_q$ | vehicle mass, pitch inertia | `m_q`, `J` | |
| $m_b$ | payload mass | `m_L` | |
| $\ell$ | rope length | `l` | |
| $M := m_q + m_b$ | total mass | `M` | |
| $r_q = [y_q, z_q]^\top$, $v_q$ | vehicle position, velocity | `p_y, p_z, v_y, v_z` | |
| $\theta_q$, $\omega_q$ | pitch, pitch rate | `theta`, `omega` | |
| $\varphi$ | rope angle, $\varphi = \operatorname{atan2}(y_q - y_b, z_q - z_b)$ | `phi` (code), **$\varphi = -\phi_{\rm code}$** at $d = 0$ | |
| $\hat u(\varphi) = [\sin\varphi, \cos\varphi]^\top$ | rope direction, load to vehicle | $-q$ | |
| $\hat t(\varphi) = [\cos\varphi, -\sin\varphi]^\top$ | tangent | $t$ (unchanged) | |
| $T^\varphi$ | rope tension | `T` | |
| $u = [f_q, \tau_q]^\top$ | thrust, torque | `(f, tau)` | |
| $F = f_q[-\sin\theta_q, \cos\theta_q]^\top$ | the thrust as a world force (the MRAC's virtual input) | `U = [F_y, F_z, tau]` | |
| $x^d$ (task "reference state") | the task target | task reference `p_r(t)` | **not** Guha's $x_r$ |
| the nominal vehicle with the 4 kg / 2 m payload under PID+FF + damper | the **reference model** | | $x_r$, $u_r$ |
| $\Lambda = \operatorname{diag}(m_{q,r}/m_q,\ M_r/M)$ | effectiveness (per channel) | | $\lambda$ (scalar in Guha) |
| $A_H = A_r - B_r K_e^\top$, $P A_H + A_H^\top P = -Q$ | error matrix and its Lyapunov pair | | $A_H$, $P$, $Q$ |
| $\Gamma_x, \Gamma_u$ | adaptation gains | | $\Gamma_x, \gamma_u$ |

Two roles must never be confused: the **task state** $x^d$ is where the vehicle should go; the **reference model** $x_r$ is a simulated nominal vehicle that the real vehicle is asked to imitate.

### 4.1 Worked example: converting the linearised swing row from code angle to residual.tex

> **Where this comes from.**
> - *Source:* the rope-angle sign map in the notation note §3, and the linearised outer model of MQP §6.14.2 (taut hover linearisation of the plant of MQP §5.3-5.5).
> - *Read first:* nothing; this section only needs the geometry below.
> - *Assumptions:*
>   1. Cable attached at the vehicle centre of mass ($d = 0$).
>   2. Taut rope, hover: $T \approx m_b g$, small $\phi$.
>   3. Planar motion, $z$ up; $\ddot y$ is the vehicle's horizontal acceleration, $\ddot z$ its vertical acceleration.

**Geometry (code).** $r_b = r_q + \ell\,q(\phi_c)$, $q = [\sin\phi_c, -\cos\phi_c]^\top$, tangent $t = \dfrac{dq}{d\phi_c} = [\cos\phi_c, \sin\phi_c]^\top$. I write $\phi_c := \phi_{\rm code}$ and $\varphi$ for the residual.tex angle.

**Step 1. Swing equation in code angle** (MQP §5.5 at $d = 0$, projection of the payload balance on $t$):
$$\ell\,\ddot\phi_c=-g\sin\phi_c-t^\top\ddot r_q=-g\sin\phi_c-\big(\cos\phi_c\,\ddot y+\sin\phi_c\,\ddot z\big).\qquad(1)$$

**Step 2. Linearise at hover.** $\sin\phi_c\approx\phi_c$, $\cos\phi_c\approx1$, and $\sin\phi_c\,\ddot z\approx 0$ because it is a product of two small quantities:
$$\ell\,\ddot\phi_c=-g\,\phi_c-\ddot y.\qquad(2)$$

**Step 3. Vehicle horizontal row in code angle.** The rope pulls the vehicle toward the payload with force $T q$ (MQP §5.3). Its horizontal part is $T\sin\phi_c\approx m_b g\,\phi_c$ (tension $T \approx m_b g$ at hover, MQP §5.7):
$$m_q\,\ddot y=F_y+m_b g\,\phi_c.\qquad(3)$$

**Step 4. Substitute $\phi_c=-\varphi$, so $\ddot\phi_c=-\ddot\varphi$:**
$$\text{(2):}\quad -\ell\,\ddot\varphi=g\,\varphi-\ddot y\ \Longrightarrow\ \boxed{\ell\,\ddot\varphi=\ddot y-g\,\varphi}\quad\text{(multiply by }-1\text{)}$$
$$\text{(3):}\quad m_q\,\ddot y=F_y+m_b g(-\varphi)\ \Longrightarrow\ \boxed{m_q\,\ddot y=F_y-m_b g\,\varphi}$$

**Step 5. Compare** with the residual.tex-sign linear model of the notation note §8.2: $m_q\ddot y_q=F_y-m_b g\varphi$ and $\ell\ddot\varphi=\ddot y_q-g\varphi$. They agree.

**Why some terms flip and others do not.** Under $\phi_c=-\varphi$ every quantity is either *odd* in the angle ($\phi_c$, $\dot\phi_c$, $\ddot\phi_c$, $\sin\phi_c$) or *even* ($\cos\phi_c$, and the quantities $\ddot y$, $\ddot z$, $F_y$, $g$, which do not depend on the rope angle). The flip rule is about the **coefficient that links two quantities**:

- Two quantities of the **same parity** (for example $\ddot\varphi$ and $\varphi$, both odd): the coefficient is **unchanged**. In (2) the restoring term is $-g/\ell$ times the angle; after the substitution it is still $-g/\ell$ times $\varphi$.
- Two quantities of **opposite parity** (for example $\ddot\varphi$ and $\ddot y$, or $\ddot y$ and $\varphi$): the coefficient **changes sign**. In (2) the $\ddot y$ coefficient is $-1/\ell$; in the residual.tex row it is $+1/\ell$. In (3) the coupling $+m_b g$ becomes $-m_b g$.

The same reasoning on the nonlinear row (1) gives $\ell\ddot\varphi=-g\sin\varphi+\hat t^\top\ddot r_q$ with $\hat t=[\cos\varphi,-\sin\varphi]^\top$: the $\sin$ terms flip, the $\cos$ term does not. This is the rule "odd functions of the angle flip; $\cos\varphi$, the tension and the tangent do not" of the notation note. The cell below checks all of this symbolically.

In [1]:
import sympy as sp

g, ell, m_q, m_b = sp.symbols("g ell m_q m_b", positive=True)
phi, y_dd, z_dd, F_y, phi_dd = sp.symbols("phi y_dd z_dd F_y phi_dd")
phi_c, phi_c_dd = -phi, -phi_dd          # code angle = minus residual.tex angle (d = 0)

# 1) Linearised rows in CODE angle (MQP 6.14.2), with phi_c = -phi substituted, against the residual.tex rows
swing_code = ell*phi_c_dd - (-g*phi_c - y_dd)          # l phi_c'' = -g phi_c - y''
swing_tex  = ell*phi_dd - (y_dd - g*phi)               # l phi''   =  y'' - g phi
veh_code   = m_q*y_dd - (F_y + m_b*g*phi_c)            # m_q y'' = F_y + m_b g phi_c
veh_tex    = m_q*y_dd - (F_y - m_b*g*phi)              # m_q y'' = F_y - m_b g phi
print("swing row, code row (phi_c=-phi) + tex row =", sp.expand(swing_code + swing_tex), " (0: the same equation, written with a factor -1)")
print("vehicle row, code row (phi_c=-phi) - tex row =", sp.expand(veh_code - veh_tex), " (0: identical)")

# 2) Nonlinear swing row: sin is odd, cos is even
swing_nl_code = -g*sp.sin(phi_c) - (sp.cos(phi_c)*y_dd + sp.sin(phi_c)*z_dd)        # l phi_c''
swing_nl_tex  = -g*sp.sin(phi)   + (sp.cos(phi)*y_dd - sp.sin(phi)*z_dd)            # l phi''
print("nonlinear swing row, (-code rhs) - tex rhs =", sp.simplify(-swing_nl_code - swing_nl_tex), " (0: sin terms flip, cos terms do not)")

# 3) With an offset d != 0 the simple map fails:
#    phi_tex = atan2(-d sin(theta) - l sin(phi_c), d cos(theta) + l cos(phi_c))   (notation note, section 3)
def phi_tex(phi_c_val, theta, d, l=2.0):
    return sp.atan2(-d*sp.sin(theta) - l*sp.sin(phi_c_val), d*sp.cos(theta) + l*sp.cos(phi_c_val))
for d, th, pc in [(0.0, 0.2, 0.5), (0.3795, 0.0, 0.5), (0.3795, 0.2, 0.0)]:
    print(f"d={d}, theta={th}, phi_code={pc}:  phi_tex = {float(phi_tex(pc, th, d)):+.4f},  -phi_code = {-pc:+.4f}")

swing row, code row (phi_c=-phi) + tex row = 0  (0: the same equation, written with a factor -1)
vehicle row, code row (phi_c=-phi) - tex row = 0  (0: identical)
nonlinear swing row, (-code rhs) - tex rhs = 0  (0: sin terms flip, cos terms do not)
d=0.0, theta=0.2, phi_code=0.5:  phi_tex = -0.5000,  -phi_code = -0.5000
d=0.3795, theta=0.0, phi_code=0.5:  phi_tex = -0.4222,  -phi_code = -0.5000
d=0.3795, theta=0.2, phi_code=0.0:  phi_tex = -0.0318,  -phi_code = -0.0000


**Why the simple rule fails at $d\neq0$.** With the cable attached at a point $d\,n(\theta)$ away from the centre of mass, the residual.tex angle is the direction of the vector from the payload to the **vehicle centre of mass**, while the code angle is the direction of the rope from the **attachment point**. The two differ by the offset, and the offset rotates with the pitch $\theta$:
$$\varphi_{\rm tex}=\operatorname{atan2}\!\big(-d\sin\theta-\ell\sin\phi_c,\ d\cos\theta+\ell\cos\phi_c\big).$$
Setting $d=0$ gives $\varphi_{\rm tex}=\operatorname{atan2}(-\ell\sin\phi_c,\ \ell\cos\phi_c)=-\phi_c$, the simple flip. For $d\ne0$ the map depends on $\theta$ and is nonlinear in $\phi_c$: in the last two lines of the output, $d = 0.3795$ m with $\theta = 0$ gives $\varphi_{\rm tex}=-0.4222\neq-0.5$, and a nonzero pitch alone ($\phi_c = 0$, $\theta = 0.2$) gives $\varphi_{\rm tex}=-0.0318\ne0$. That is why NB05-NB09 stay in the code angle at $d = 0.3795$ m and do not apply the sign flip.

**What this example does and does not show.** It verifies the sign map for the linearised and nonlinear swing rows at $d = 0$, and shows by numbers that the map is not a sign flip at $d\ne0$. It is an algebra and notation check, not a model-validity result.

## 5. A complete scalar MRAC derivation (the template for all later derivations)

Every derivation in NB01-NB13 is written in the same order: a **Where this comes from** box, **numbered steps** with a short justification beside each line, a plain-language **intuition**, and a **What this does / does not show** box. This section shows that pattern on the simplest case: a scalar plant with two unknown parameters (NB01 and NB02 develop it in detail).

> **Where this comes from.**
> - *Source:* NB02 §02 (scalar Lyapunov derivation; the Lyapunov redesign is due to Parks, 1966, as cited in NB02); the matching conditions and error equation of NB01 §02.
> - *Read first:* NB01 (reference model, matching conditions, error equation).
> - *Assumptions:*
>   1. Plant $\dot x = A x + B u$, with scalar $A$, $B$ **unknown constants**, $B\neq 0$ of **known sign**.
>   2. Reference model $\dot x_m = A_m x_m + B_m r$ with $A_m<0$ (Hurwitz in the scalar case), $B_m$ known, $r(t)$ bounded and piecewise continuous.
>   3. Control $u = K_r r - K_x x$ with time-varying gains.
>   4. **Ideal gains** $K_x^*$, $K_r^*$ exist and are constant: $A - BK_x^* = A_m$ and $BK_r^* = B_m$ (matching conditions).
>   5. Continuous time; no saturation, noise, delay or unmodelled dynamics.

**Step 1. Tracking error and its derivative.** Define $e := x - x_m$. Then
$$\dot e=\dot x-\dot x_m=Ax+B(K_r r-K_x x)-A_m x_m-B_m r\qquad\text{(substitute plant, control and reference)}.$$

**Step 2. Add and subtract $A_m x$** to pair the $x_m$ term with an $x$ term:
$$\dot e=A_m(x-x_m)+(A-BK_x-A_m)\,x+(BK_r-B_m)\,r=A_m e+(A-BK_x-A_m)\,x+(BK_r-B_m)\,r.$$
Check: $A_m x-A_m x_m=A_m e$, and the remaining $x$ and $r$ terms are the two brackets.

**Step 3. Use the matching conditions** $A-A_m=BK_x^*$ and $B_m=BK_r^*$ (assumption 4):
$$\dot e=A_m e+B(K_x^*-K_x)\,x+B(K_r-K_r^*)\,r=A_m e-B\tilde K_x\,x+B\tilde K_r\,r,\qquad \tilde K:=K-K^*.$$
Because $K^*$ is constant, $\dot{\tilde K}=\dot K$ (used in Step 5).

**Step 4. Lyapunov candidate.** With adaptation gains $\gamma_x,\gamma_r>0$,
$$V=\tfrac12 e^2+\frac{|B|}{2\gamma_x}\tilde K_x^2+\frac{|B|}{2\gamma_r}\tilde K_r^2 .$$
It is positive definite in $(e,\tilde K_x,\tilde K_r)$. The factor $|B|$ is there so that the unknown $B$ cancels in Step 6; $\gamma$ is in the denominator, so a larger gain makes parameter error cheaper and lets the gain move faster.

**Step 5. Differentiate** (chain rule; $\dot{\tilde K}=\dot K$):
$$\dot V=e\dot e+\frac{|B|}{\gamma_x}\tilde K_x\dot K_x+\frac{|B|}{\gamma_r}\tilde K_r\dot K_r .$$
Substitute $\dot e$ from Step 3:
$$\dot V=A_m e^2-B\,e\,\tilde K_x x+B\,e\,\tilde K_r r+\frac{|B|}{\gamma_x}\tilde K_x\dot K_x+\frac{|B|}{\gamma_r}\tilde K_r\dot K_r .$$

**Step 6. Choose the update laws that cancel the parameter terms.** Group by $\tilde K_x$ and $\tilde K_r$:
$$\dot V=A_m e^2+\tilde K_x\Big(-B e x+\frac{|B|}{\gamma_x}\dot K_x\Big)+\tilde K_r\Big(B e r+\frac{|B|}{\gamma_r}\dot K_r\Big).$$
Setting each bracket to zero and using $B/|B|=\operatorname{sgn}(B)$:
$$\boxed{\dot K_x=\operatorname{sgn}(B)\,\gamma_x\,e\,x,\qquad \dot K_r=-\operatorname{sgn}(B)\,\gamma_r\,e\,r .}$$
Check of the first cancellation: $-Bex+\frac{|B|}{\gamma_x}\operatorname{sgn}(B)\gamma_x ex=-Bex+Bex=0$ because $|B|\operatorname{sgn}(B)=B$. The laws contain neither $A$ nor $B$ (only its sign), which is what makes them implementable.

**Step 7. The derivative that remains:**
$$\dot V=A_m e^2=-|A_m|\,e^2\le 0 .$$
It is negative when $e\ne0$ but zero when $e=0$ with parameter errors still nonzero: $\dot V$ is **negative semidefinite** in $(e,\tilde K_x,\tilde K_r)$, not negative definite.

**Step 8. Conclusion (conditional, Barbalat).**
1. $V\ge0$ and $\dot V\le0$, so $V(t)\le V(0)$: $e$, $\tilde K_x$, $\tilde K_r$ are bounded.
2. $\int_0^\infty e^2\,dt\le V(0)/|A_m|$.
3. If $r$ is bounded then $x_m$ is bounded, so $x=e+x_m$ is bounded, so $\dot e=A_me-B\tilde K_xx+B\tilde K_rr$ is bounded, so $\ddot V=2A_m e\dot e$ is bounded and $\dot V$ is uniformly continuous.
4. Barbalat's lemma then gives $\dot V\to0$, that is $e(t)\to0$.

**Intuition.** $V$ measures how far the system is from "ideal": tracking error plus wrong gains. The update laws are chosen so that the gains move exactly in the direction that cancels the term by which the wrong gains push the error. What is left is the reference model's own decay $A_m e^2$. The gains stop moving when the error stops, not necessarily when they have reached $K^*$.

> **What this does / does not show.**
> - It shows tracking-error convergence $e\to0$ for the scalar matched continuous-time model, **conditional** on assumptions 1-5 and the boundedness used in Step 8.
> - It does **not** show parameter convergence ($\tilde K\to0$): that needs persistent excitation, which is a separate condition.
> - It does **not** cover noise, delay, saturation, sampling or unmodelled dynamics; a real controller needs robust modifications ($\sigma$-modification, projection; see the glossary).
> - It is a derivation for a scalar plant. The MIMO version (NB03) and the quadrotor-payload versions (NB10-NB13) need their own premises, listed in their boxes.

## 6. How to read the boxes in the other notebooks

| Box | What it tells you |
|---|---|
| **Where this comes from** | the source (MQP section with its name, paper equation or theorem, or textbook result), the notebook to read first, and the numbered assumptions |
| **Scope banner** (`EXPLORATORY / offline / PROXY`; the $d$ and angle-convention banner) | which model the numbers belong to; do not carry a result outside that scope |
| **Provenance** | which source commit produced a cached result, and whether the current code differs (then the numbers are historical observations) |
| **What this does / does not show** | the claim limit of the derivation or result; it is part of the result |

Claim limits are never weakened between notebooks: a statement that is conditional here stays conditional wherever it is reused.